## Fast Food Density & Income Pipeline
CSE 3104 — Data Manipulation & Management: Final Project 

Connects ACS 5-Year Census income data to McDonald's locations by zip code.

In [ ]:
# Imports
import pandas as pd
import numpy as np

In [ ]:
# Load Census income data
# skiprows=[1] drops the Census description row; row 0 has the column codes
census = pd.read_csv("ACSDT5Y2024.B19013-Data.csv", skiprows=[1])

census["zip_code"] = census["NAME"].str.extract(r"(\d{5})") #We want to standardized the zipcodes into 5 numbers
census["median_income"] = pd.to_numeric(census["B19013_001E"], errors="coerce")
census = census[["zip_code", "median_income"]].copy()

census.head(3)

,zip_code,median_income
0,00601,19454.0
1,00602,21420.0
2,00603,20933.0


In [ ]:
# Impute missing income using state-level average
# First 2 digits of zip code = state proxy (like for 63 = Missouri)
census["state_prefix"] = census["zip_code"].str[:2]

missing_before = census["median_income"].isna().sum()

census["median_income"] = census.groupby("state_prefix")["median_income"].transform(
    lambda x: x.fillna(x.mean())
)

missing_after = census["median_income"].isna().sum()
print(f"Imputed {missing_before - missing_after} values ({missing_before} → {missing_after} missing)")

census = census.drop(columns="state_prefix")
census.head(3)

Imputed 3358 values (3358 → 0 missing)


,zip_code,median_income
0,00601,19454.0
1,00602,21420.0
2,00603,20933.0


In [ ]:
# Load McDonald's location data
# Kaggle dataset: jacopomazzoni/mcdonalds-locations-2025
mcdonalds = pd.read_csv("mcdonalds_locations.csv")

# Rename zipcode -> zip_code for consistency, then zero-pad to 5 digits
mcdonalds = mcdonalds.rename(columns={"zipcode": "zip_code"})
mcdonalds["zip_code"] = mcdonalds["zip_code"].astype(str).str.zfill(5).str[:5]

print(f"Loaded {len(mcdonalds):,} McDonald's locations")
mcdonalds[["zip_code", "city", "state"]].head(3)

Loaded 13,418 McDonald's locations


,zip_code,city,state
0,33040,Key West,FL
1,33050,Marathon,FL
2,33070,Tavernier,FL


In [ ]:
# Count McDonald's per zip code
store_counts = (
    mcdonalds
    .groupby("zip_code")
    .size()
    .reset_index(name="store_count")
)

print(f"Unique zips with a McDonald's: {len(store_counts):,}")
store_counts.head(3)

Unique zips with a McDonald's: 9,175


,zip_code,store_count
0,01001,1
1,01007,1
2,01008,2


In [ ]:
# Join income data to store counts
# Left join keeps all 33k Census zips; unmatched zips get store_count = 0
merged = census.merge(store_counts, on="zip_code", how="left")
merged["store_count"] = merged["store_count"].fillna(0).astype(int)

matched = (merged["store_count"] > 0).sum()
unmatched = len(store_counts) - matched

print(f"Zips matched to a McDonald's: {matched:,}")
print(f"McDonald's locations outside Census zips: {unmatched:,}")

#unmatched_percentage = (matched / unmatched) * 100
#print(unmatched_percentage)

merged.head(5)

Zips matched to a McDonald's: 9,082
McDonald's locations outside Census zips: 93
9765.591397849463


,zip_code,median_income,store_count
0,00601,19454.0,0
1,00602,21420.0,0
2,00603,20933.0,0
3,00606,20992.0,0
4,00610,24496.0,0


In [ ]:
# Add density per 10k residents
# Requires population per zip — load from ACS DP05 demographic dataset
# Columns needed: zip_code, total_population
demographics = pd.read_csv("ACSDP5Y2024.DP05-Data.csv", skiprows=[1])
demographics["zip_code"] = demographics["NAME"].str.extract(r"(\d{5})")
demographics["total_population"] = pd.to_numeric(demographics["DP05_0001E"], errors="coerce")
demographics = demographics[["zip_code", "total_population"]]
demographics.head(3)

,zip_code,total_population
0,NaN,334922499


In [ ]:
# Export final merged table
OUT_PATH = "National_Income_FastFood_Merged.csv"
merged.to_csv(OUT_PATH, index=False)
print(f"Saved {len(merged):,} rows to {OUT_PATH}")

Saved 33,772 rows to National_Income_FastFood_Merged.csv
